In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

In [3]:
eval_iters = 200
batch_size = 32
eval_interval = 500
block_size = 8
max_iters = 5000
learning_rate = 3e-4
n_embd = 32
n_head = 4
n_layer = 3
dropout = 0.2

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print ("device: ", device)

device:  cpu


In [5]:
with open ("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [6]:
print("length of dataset in chars: ", len(text))

length of dataset in chars:  1115394


In [7]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [8]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("characters:", "".join(chars))
print("vocabulary size: ", vocab_size)

characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocabulary size:  65


In [9]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join (itos[i] for i in l)

In [10]:
print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [11]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:100])

torch.Size([1115394]) <built-in method type of Tensor object at 0x1118d08c0>
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [12]:
n = int(0.9 * len (data))

train_data = data[:n]
val_data = data[n:]

In [13]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb =  get_batch("train")

print ("inputs")
print(xb.shape)
print(xb)

print ("targets")
print(yb.shape)
print(yb)

print ("-----------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print (f"when the input is {context}, the target is {target}")


inputs
torch.Size([32, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46],
        [43,  1, 51, 39, 63,  1, 40, 43],
        [58, 46, 43,  1, 43, 39, 56, 57],
        [39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1],
        [39, 51,  1, 39, 44, 56, 39, 47],
        [17, 24, 21, 38, 13, 14, 17, 32],
        [ 1, 39, 52, 42,  1, 45, 43, 50],
        [ 1, 58, 46, 39, 58,  1, 42, 53],
        [ 1, 61, 53, 59, 50, 42,  1, 21],
        [59, 57, 40, 39, 52, 42,  1, 40],
        [52, 42,  8,  0,  0, 23, 21, 26],
        [45, 53, 42, 57,  0, 23, 43, 43],
        [52,  1, 61, 39, 57,  1, 51, 53],
       

In [14]:
class Head (nn.Module):
    
    def __init__ (self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)  
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**(-0.5)
        wei = wei.masked_fill (self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax (wei, dim = -1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [15]:
class MultiHeadAttention(nn.Module):

    def __init__ (self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList ([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear (n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out

In [16]:
class FeedForward(nn.Module):

    def __init__ (self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
            )

    def forward(self, x):
        return self.net(x)

In [17]:
class Block(nn.Module):

    def __init__ (self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward (self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [18]:
class BigramLanguageModel(nn.Module):

    def __init__ (self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential (* [Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward (self, idx, targets = None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange (T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate (self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits [:, -1, :]
            probs = F.softmax (logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

model = BigramLanguageModel()
model = model.to(device)

In [19]:
total_params = sum(p.numel() for p in model.parameters())
print("model parameters count: ", total_params)

model parameters count:  42369


In [20]:
logits, loss = model(xb, yb)

print(logits.shape)
print(loss)

torch.Size([256, 65])
tensor(4.2951, grad_fn=<NllLossBackward0>)


In [21]:
idx = torch.zeros((1, 1), dtype=torch.long)
print (decode(model.generate(idx, max_new_tokens=100)[0].tolist()))


yt&Lkg?g-xe$,EqFqZ;EbAhkfR.H?
E3!d
b-OfrfEDQjXIZuAjH$ERk-$spHEhiy,N z
$gRC:,rZAq3mEmZUHQ'Oaik'Obz,Ez


In [22]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [23]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range (eval_iters):
            X, Y = get_batch(split)
            _, loss = model (X, Y)
            losses [k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out        

In [24]:
for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses["train"]:.4f}, val loss = {losses["val"]:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.3514, val loss = 4.3565
step 500: train loss 2.7530, val loss = 2.7581
step 1000: train loss 2.5198, val loss = 2.5302
step 1500: train loss 2.4413, val loss = 2.4654
step 2000: train loss 2.3980, val loss = 2.3974
step 2500: train loss 2.3541, val loss = 2.3650
step 3000: train loss 2.3359, val loss = 2.3424
step 3500: train loss 2.2943, val loss = 2.3116
step 4000: train loss 2.2707, val loss = 2.2988
step 4500: train loss 2.2599, val loss = 2.2844


In [25]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print (decode(model.generate(context, max_new_tokens=500)[0].tolist()))



DMURAWPTUS:
Thim shifs, Teage, ar sefrile natspakelf?

Is:
Rocte srethhoulse thees hito thhe feain the
O houls gavind sadS: fand cous fur:
Wit semed tousild klere,
Es
Nthing:
Eidg of hip eafornen. quretis datm al hat ich dilld by, ofou Jorou Ives loy thourd:
Hedimes tarto hencowe so meat timkhinges tard't fonow sehou dod san's
To thomid
Shall hal
ICat cobunef!
BORD:
Go Proment, forgreletore fong we heretset youllt by wivale bE His nom,
Pratir, ule kee forseerotcen bust matheadit ly wis yot of l
